In [ ]:
pip install pdfplumber scikit-learn nltk

In [ ]:
import pdfplumber
import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
# Load Skills Database


def load_skills(file_path="skills.txt"):
    
    with open(file_path, "r") as f:
        skills = [line.strip().lower() for line in f]
        
    return skills


skills_db = load_skills()


In [ ]:
def extract_resume_text(file_path):

    text = ""

    with pdfplumber.open(file_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text

    return text


In [ ]:
# Text Preprocessing


stop_words = set(stopwords.words("english"))

def preprocess(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [ ]:
# Skill Extraction

def extract_skills(text, skills_db):

    text = text.lower()

    found_skills = []

    for skill in skills_db:

        if skill in text:
            found_skills.append(skill)

    return found_skills

In [ ]:
def find_missing_skills(resume_skills, job_skills):

    missing = []

    for skill in job_skills:

        if skill not in resume_skills:
            missing.append(skill)

    return missing


In [ ]:
# Resume Analysis


def analyze_resume(resume_path, job_description):

    print("\nExtracting resume text...\n")

    resume_text = extract_resume_text(resume_path)

    # Preprocess
    resume_clean = preprocess(resume_text)
    job_clean = preprocess(job_description)

    # Extract Skills
    resume_skills = extract_skills(resume_text, skills_db)
    job_skills = extract_skills(job_description, skills_db)

    # Skill text
    resume_skill_text = " ".join(resume_skills)
    job_skill_text = " ".join(job_skills)

    # TF-IDF similarity
    documents = [resume_skill_text, job_skill_text]

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(documents)

    similarity = cosine_similarity(tfidf_matrix[0], tfidf_matrix[1])

    similarity_score = similarity[0][0]

    # Matched skills
    matched_skills = set(resume_skills) & set(job_skills)

    match_count = len(matched_skills)
    total_required = len(job_skills)

    skill_score = match_count / total_required if total_required > 0 else 0

    # Match Category
    if skill_score < 0.25:
        category = "High Improvement Required"

    elif skill_score < 0.50:
        category = "Moderate Alignment"

    elif skill_score < 0.75:
        category = "Strong Alignment"

    else:
        category = "Excellent Match"

    # Missing skills
    missing_skills = find_missing_skills(resume_skills, job_skills)

In [ ]:
# Output

    print("\n========== RESUME ANALYSIS ==========\n")

    print("Similarity Score:", round(similarity_score, 2))

    print("Match Level:", category)

    print("\nMatched Skills:")
    for skill in matched_skills:
        print("-", skill)

    print("\nMissing Skills:")
    for skill in missing_skills:
        print("-", skill)

    print("\nSuggestions for Improvement:\n")

    for skill in missing_skills:

        if skill == "aws":
            print("- Learn cloud platforms such as AWS or Azure.")

        elif skill == "docker":
            print("- Gain experience with containerization tools like Docker.")

        elif skill == "sql":
            print("- Improve database querying skills using SQL.")

        else:
            print(f"- Consider learning {skill} to improve job alignment.")